# Search lookup magics

Use **`%%cribl_save_search_lookup`** / **`%%cribl_load_search_lookup`** / **`%%cribl_delete_search_lookup`** for Search lookup CSV files under the **`default_search`** API group (same as `%%cribl_search`).

**Limits:** lookups are capped at **10,000 rows**. Use **`replace=true`** on save to overwrite. Pick **distinct filenames** (for example `notebook_app_lookup_demo.csv`) so you do not clash with shared lookups.

This notebook also shows **querying lookup contents** with the `$vt_lookups` virtual table, and **creating then deleting** a separate lookup using the **`cribl-control-plane`** stack plus **`httpx`** (the SDK's HTTP dependency) calling the same REST endpoints as the magics.

## Setup

Run **once** before other code cells. Matches `Cribl_Python_SDK.ipynb` pins for Pyodide WASM wheels.


In [ ]:
import micropip

# Match Pyodide 0.29.x built-in stack (WASM). Update when bumping PYODIDE_RELEASE.
await micropip.install([
    'httpcore==1.0.7',
    'httpx==0.28.1',
    'pydantic-core==2.41.5',
    'pydantic==2.12.5',
    'cribl-control-plane',
])


## A) Cell magics: Search → save → search via `$vt_lookups` → load → verify


In [ ]:
%%cribl_search var=sample_df preview=false
dataset=cribl_search_sample | sort by _time desc | limit 50


In [ ]:
%%cribl_save_search_lookup notebook_app_lookup_demo.csv var=sample_df replace=true


In [ ]:
%%cribl_search var=vt_rows preview=true
dataset="$vt_lookups" lookupFile="notebook_app_lookup_demo"
| limit 25


In [ ]:
%%cribl_load_search_lookup notebook_app_lookup_demo.csv var=from_lookup


In [ ]:
print('search rows:', len(sample_df))
print('vt_lookups rows:', len(vt_rows))
print('loaded rows:', len(from_lookup))
from_lookup.head(3)


## B) Create & delete a lookup with Python (`httpx` + `cribl-control-plane`)

The preview SDK does not yet expose Search lookup helpers. This cell uses **`httpx`** for `PUT`/`POST`/`DELETE` on `/m/default_search/system/lookups` (same flow as the magics) and a quick **`CriblControlPlane.health`** call so both stacks run together.


In [ ]:
import json
import os
from urllib.parse import quote

import httpx
from cribl_control_plane import CriblControlPlane

base = os.environ['CRIBL_API_URL'].rstrip('/')
group = 'default_search'
lookup_name = 'notebook_app_sdk_lookup_demo.csv'
csv_body = 'join_key,label\nalpha,from_httpx\nbeta,second_row\n'

print('SDK health ping:', end=' ')
with CriblControlPlane(server_url=base) as ccp:
    print(ccp.health.get())

with httpx.Client(timeout=60.0) as client:
    r_put = client.put(
        f'{base}/m/{group}/system/lookups',
        params={'filename': '__nb_sdk_tmp.csv'},
        content=csv_body.encode('utf-8'),
        headers={'Content-Type': 'text/csv'},
    )
    r_put.raise_for_status()
    tmp_name = r_put.json()['filename']
    r_post = client.post(
        f'{base}/m/{group}/system/lookups',
        json={'id': lookup_name, 'fileInfo': {'filename': tmp_name}},
        headers={'Content-Type': 'application/json'},
    )
    r_post.raise_for_status()
    qn = quote(lookup_name, safe='')
    r_del = client.delete(f'{base}/m/{group}/system/lookups/{qn}')
    if r_del.status_code not in (200, 204, 404):
        r_del.raise_for_status()
print('httpx create/delete finished for', lookup_name)


## Cleanup (cell magics)

Deletes the lookup created in section **A** (`notebook_app_lookup_demo.csv`). Section **B** already removed its own file.


In [ ]:
%%cribl_delete_search_lookup notebook_app_lookup_demo.csv
